# Reconhecimento de rostos em imagens estáticas (OpenCV)

**Dupla:** Márcio Augusto e Daniel Conceição

Atividade prática de Visão Computacional. Rode as células **em ordem**.

1. **Instalar e conferir o OpenCV.** O código precisa da versão 4.x.
2. **Carregar as fotos:** `foto1.jpg` (uma pessoa), `foto2.jpg` (grupo de 4) e `foto3.jpg` (multidão).
   O notebook procura as fotos na pasta **3 - Fotos** do Google Drive. Se não achar, pede para você enviar.
3. **Código original do PDF.**
4. **Teste de limites, parâmetros e mudança visual.** É o desafio da atividade.
5. **Testes extras:** a multidão e a aplicação prática (LGPD).

## 0. Preparação

> ⚠️ **No OpenCV 5 o `CascadeClassifier` saiu do pacote principal.** Com ele, o código do PDF dá
> `AttributeError: module 'cv2' has no attribute 'CascadeClassifier'`. Por isso forçamos a versão 4.x.
> Se a célula de baixo instalar algo, vá em **Ambiente de execução → Reiniciar sessão** e continue a partir da célula seguinte.

In [ ]:
!pip install -q "opencv-python<5" "opencv-python-headless<5" "opencv-contrib-python<5"

In [ ]:
import os
import cv2

print('OpenCV', cv2.__version__)
assert hasattr(cv2, 'CascadeClassifier'), 'OpenCV 5 detectado: reinicie a sessão e rode a partir desta célula.'

# No Colab, cv2.imshow não abre janela: usamos o cv2_imshow do próprio Colab
try:
    from google.colab.patches import cv2_imshow
except ImportError:  # rodando no computador
    def cv2_imshow(img):
        cv2.imshow('Reconhecimento - Pressione qualquer tecla para fechar', img)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

In [ ]:
# Procura foto1.jpg, foto2.jpg e foto3.jpg: na pasta fotos/, na pasta do Google Drive ou pede o envio
FOTOS = ['foto1.jpg', 'foto2.jpg', 'foto3.jpg']
PASTA_DRIVE = '/content/drive/MyDrive/Atividade Visão Computacional - Reconhecimento de Rostos/3 - Fotos'


def tem_fotos(pasta):
    return all(os.path.exists(os.path.join(pasta, f)) for f in FOTOS)


PASTA = next((p for p in ('fotos', '.') if tem_fotos(p)), None)
if PASTA is None:
    try:
        from google.colab import drive, files
        drive.mount('/content/drive')  # o Colab pede autorização para acessar o seu Drive
        if tem_fotos(PASTA_DRIVE):
            PASTA = PASTA_DRIVE
        else:
            print('Não achei as fotos no Drive. Envie foto1.jpg, foto2.jpg e foto3.jpg:')
            files.upload()
            PASTA = '.'
    except ImportError:
        PASTA = '.'

print('Pasta das fotos:', PASTA)
print('Fotos encontradas:', [f for f in FOTOS if os.path.exists(os.path.join(PASTA, f))])

## 1. Código original do PDF

O código foi transcrito como está no PDF, com uma única troca: `cv2.imshow` virou `cv2_imshow` (Colab).

Repare que `cv2.resize(imagem, (800, 600))` **achata** a foto 1, que é um retrato: ela vira paisagem.
Nas próximas células o redimensionamento mantém a proporção.

In [ ]:
# Carrega o classificador pré-treinado para reconhecimento de rostos
classificador_rosto = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Carrega a imagem estática
imagem = cv2.imread(os.path.join(PASTA, 'foto1.jpg'))

# Converte a imagem para tons de cinza (simplifica o processamento)
imagem_cinza = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)

# Realiza o processo de classificação e detecta os rostos na imagem
rostos = classificador_rosto.detectMultiScale(imagem_cinza, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

# Desenha um retângulo verde ao redor de cada rosto detectado
for (x, y, w, h) in rostos:
    cv2.rectangle(imagem, (x, y), (x+w, y+h), (0, 255, 0), 2)

# Redimensiona a imagem para caber melhor na tela caso seja muito grande
imagem_redimensionada = cv2.resize(imagem, (800, 600))

# Mostra o resultado final na tela
cv2_imshow(imagem_redimensionada)
print(len(rostos), 'rosto(s) detectado(s)')

## 2. Funções para não repetir código

Em vez de editar e rodar o código 9 vezes por foto, usamos duas funções:
- `detectar` faz a classificação;
- `mostrar` desenha os retângulos e exibe a foto mantendo a proporção.

In [ ]:
def carregar(nome):
    imagem = cv2.imread(os.path.join(PASTA, nome))
    if imagem is None:  # imread não avisa quando o arquivo não existe: devolve None
        raise FileNotFoundError(f'Não achei {nome}. Confira o nome e se a foto foi enviada.')
    return imagem


def detectar(imagem, scale=1.1, vizinhos=5):
    if isinstance(imagem, str):
        imagem = carregar(imagem)
    cinza = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)
    return classificador_rosto.detectMultiScale(cinza, scaleFactor=scale, minNeighbors=vizinhos, minSize=(30, 30))


def mostrar(imagem, rostos, cor=(0, 255, 0), espessura=2, largura=800):
    saida = imagem.copy()
    for (x, y, w, h) in rostos:
        cv2.rectangle(saida, (x, y), (x + w, y + h), cor, espessura)
    fator = largura / saida.shape[1]  # mantém a proporção da foto
    cv2_imshow(cv2.resize(saida, None, fx=fator, fy=fator))


imagens = {nome: carregar(nome) for nome in FOTOS}

## 3. Teste de limites (parâmetros padrão: `scaleFactor=1.1`, `minNeighbors=5`)

In [ ]:
for nome, imagem in imagens.items():
    rostos = detectar(imagem)
    print(f'{nome}: {len(rostos)} detecção(ões)')
    mostrar(imagem, rostos, espessura=max(2, imagem.shape[1] // 200))

**O que observamos (conferido visualmente):**

| Foto | Rostos reais | Detectados | Acertos | Falsos positivos | Falsos negativos |
|---|---|---|---|---|---|
| foto1: uma pessoa | 1 | 1 | 1 | 0 | 0 |
| foto2: grupo de 4 | 4 | 4 | 4 | 0 | 0 |
| foto3: multidão | ~100 (estimativa) | 1 | 0 | 1 (o corpo de uma pessoa) | ~100 |

**Por que a multidão falha:**
- **Tamanho:** cada rosto tem de 10 a 20 px. O `haarcascade_frontalface_default` foi treinado com janela de **24×24 px**, então abaixo disso ele nem procura.
- **Ângulo:** o modelo é de rosto **frontal**, e muita gente está de cabeça baixa olhando o celular ou de perfil.
- **Oclusão:** óculos escuros, bonés, cabelo e outras pessoas na frente.

## 4. Modificação de parâmetros

- `scaleFactor` diz de quanto em quanto a imagem é reduzida a cada passada. Um valor **menor** (1.05) testa mais tamanhos: fica mais sensível e mais lento.
- `minNeighbors` diz quantas janelas vizinhas precisam concordar para aceitar um rosto. Um valor **maior** (8) deixa o classificador mais rigoroso.

In [ ]:
import pandas as pd

linhas = []
for nome, imagem in imagens.items():
    for scale in (1.05, 1.1, 1.2):
        for viz in (3, 5, 8):
            linhas.append({'foto': nome, 'scaleFactor': scale, 'minNeighbors': viz,
                           'detectados': len(detectar(imagem, scale, viz))})

tabela = pd.DataFrame(linhas).pivot(index=['scaleFactor', 'minNeighbors'], columns='foto', values='detectados')
tabela

**Resultado conferido visualmente.** Cada célula mostra os rostos detectados e, entre parênteses, quantos eram falsos:

| scaleFactor | minNeighbors | foto1 (1 rosto) | foto2 (4 rostos) | foto3 (~100) |
|---|---|---|---|---|
| 1.05 | 3 | 7 (6 falsos) | 6 (2 falsos) | 2 (2 falsos) |
| 1.05 | 5 | 5 (4 falsos) | 5 (1 falso) | 2 (2 falsos) |
| 1.05 | 8 | 3 (2 falsos) | 4 (0) | 1 (1 falso) |
| 1.1 | 3 | 4 (3 falsos) | 4 (0) | 2 (2 falsos) |
| **1.1** | **5 (original)** | **1 (0)** | **4 (0)** | **1 (1 falso)** |
| 1.1 | 8 | 1 (0) | 4 (0) | 0 |
| 1.2 | 3 | 2 (1 falso) | 4 (0) | 1 (1 falso) |
| 1.2 | 5 | 1 (0) | 4 (0) | 0 |
| 1.2 | 8 | 1 (0) | 4 (0) | 0 |

**Padrão de classe e rigor.** O padrão de classe é o "rosto" que o modelo aprendeu no treino, só com rostos de frente. Os parâmetros definem o quanto uma região precisa se parecer com esse padrão:
- **Valores permissivos** (1.05 e 3) inventam rostos. Na foto 1 apareceram no cabelo e na camisa. Na foto 2, a **camisa xadrez** e a calça jeans viraram rosto: xadrez é puro padrão claro/escuro, igual às características de Haar.
- **Valores rigorosos** (1.2 e 8) eliminam os falsos, mas na multidão não aprovam nada.

In [ ]:
# Os falsos positivos dos parâmetros permissivos
for nome in ('foto1.jpg', 'foto2.jpg'):
    imagem = imagens[nome]
    mostrar(imagem, detectar(imagem, 1.05, 3), cor=(0, 0, 255), espessura=max(2, imagem.shape[1] // 200))

## 5. Modificação visual: cor e espessura

O OpenCV usa **BGR**, não RGB: `(0, 0, 255)` é vermelho e `(255, 0, 255)` é magenta.

A foto 1 tem 3000 px de largura. Uma linha de espessura 2 **quase some** quando ela é reduzida para 800 px. Por isso a espessura acompanha o tamanho da foto.

In [ ]:
MAGENTA = (255, 0, 255)
for nome, imagem in imagens.items():
    espessura = max(4, imagem.shape[1] // 200)  # 15 na foto1, 4 nas outras
    mostrar(imagem, detectar(imagem), cor=MAGENTA, espessura=espessura)

## 6. Teste extra: o problema da multidão é o tamanho?

Se o problema fosse só o `minSize`, diminuir esse valor resolveria. Não resolve, porque a janela mínima do modelo é 24×24 px.
Ampliar a foto, por outro lado, faz os rostos passarem desse tamanho.

In [ ]:
foto3 = imagens['foto3.jpg']
cinza3 = cv2.cvtColor(foto3, cv2.COLOR_BGR2GRAY)
for minimo in (30, 20, 10):
    n = len(classificador_rosto.detectMultiScale(cinza3, scaleFactor=1.1, minNeighbors=5, minSize=(minimo, minimo)))
    print(f'minSize={minimo}x{minimo}: {n} detecção(ões)')

ampliada = cv2.resize(foto3, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
rostos_amp = detectar(ampliada)
print(f'\nFoto 3 ampliada 3x: {len(rostos_amp)} detecções')
mostrar(ampliada, rostos_amp, espessura=4, largura=1200)

Com a foto ampliada 3×, ele encontrou **25 detecções: 21 rostos certos e 4 erros**. Os erros foram:
- uma cintura;
- um tronco inteiro;
- um rosto contado duas vezes;
- **óculos escuros apoiados no alto da cabeça**, que pareceram dois olhos para as características de Haar.

## 7. Aplicação prática: borrar rostos (LGPD)

Pela LGPD, a imagem de uma pessoa é dado pessoal. Com o mesmo detector dá para **borrar os rostos automaticamente** antes de publicar fotos de eventos ou escolas.

In [ ]:
anonima = imagens['foto2.jpg'].copy()
for (x, y, w, h) in detectar(anonima):
    anonima[y:y+h, x:x+w] = cv2.GaussianBlur(anonima[y:y+h, x:x+w], (51, 51), 30)
cv2_imshow(anonima)

## Conclusões

- **Melhor desempenho:** foto 2, com 4 de 4 rostos, perfeita em 7 das 9 combinações. Rostos de frente, luz uniforme e fundo liso.
- **Pior desempenho:** foto 3, sem nenhum rosto certo em nenhuma combinação. Os rostos são pequenos demais, estão de lado ou de cabeça baixa, e há oclusão.
- **Parâmetros:** quanto mais permissivos, mais falsos positivos. Quanto mais rigorosos, mais falsos negativos. Não existe valor perfeito para todas as fotos.
- **Limitação do método:** o Haar Cascade é rápido, mas só reconhece rostos frontais e de tamanho razoável. Para multidões, modelos de deep learning funcionam melhor.